# AR Catalog Algorithm Sensitivity Testing

The spatiotemporal DBSCAN procedure we use to create our AR catalog involves several hyperparameters. The final choices of these hyperparameters were largely motivated by our physical understanding of ARs in Antarctica (background domain knowledge), and verified 'by-eye' by tracking down known ARs from numerous case studies in the Antarctic AR literature and checking that their landfalling time, duration, etc. matches what we found in our catalog. However, there are several perturbations to these parameters that we could have made, and this notebook documents the sensitivity of various AR metrics to these perturbations, conveying the overall importance of getting certain hyperparameters "right."

Jimmy Butler, September 2025

## Hyperparameters Overview

In the clustering algorithm, there are several hyperparameters:
+ `epsilon_space`: spatial neighborhood, given in fractions of synoptic scale (1000 km)
+ `epsilon_time`: time neighborhood, given in hours
+ `minpts`: the minimum number of neighboring points to be considered a core point
+ `n_rep_pts`: the number of representative points to sample from each cluster at each time step

In the below sections, we explore the sensitivity of the clustering results to each of these hyperparameters. In our explorations and ground-truthing exercises, we found that an ideal set of hyperparameters are `epsilon_space: 0.5`, `epsilon_time: 12`, `minpts: 5`, and `n_rep_pts: 10`. We will perturb these hyperparameters individually and explore how they affect clustering results relative to this base setting.

In [1]:
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os
from pathlib import Path
os.chdir(Path(os.getcwd()).parents[3]/'packages')
from catalog_display import display_utils as d
import seaborn as sns


path = '/scratch/users/butlerj/extreme_antarctic_ars/catalog_runs/'

In [20]:
# small helper function to grab storms from dictionaries of dataframes
def process_cts(dictionary):

    res_dict = {}
    total_cts = []
    landfall_cts = []
    for param, df in dictionary.items():
        landfalling = df[df.is_landfalling]
        total_cts.append(df.shape[0])
        landfall_cts.append(landfalling.shape[0])

    res_dict = {'total_ct': total_cts, 'landfalling': landfall_cts}
    res_df = pd.DataFrame(res_dict, index=list(dictionary.keys()))

    return res_df
    

In [2]:
baseline_df = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts5_nreppts10_seed12345.h5')

## epsilon_space perturbations

In [3]:
# constructing dictionaries to organize results
space_dict = {}
space_dict['0.5'] = baseline_df
space_dict['0.25'] = pd.read_hdf(path + 'epsspace0.25_epstime12_minpts5_nreppts10_seed12345.h5')
space_dict['0.75'] = pd.read_hdf(path + 'epsspace0.75_epstime12_minpts5_nreppts10_seed12345.h5')
space_dict['1.0'] = pd.read_hdf(path + 'epsspace1.0_epstime12_minpts5_nreppts10_seed12345.h5')

In [22]:
res_df = process_cts(space_dict)
res_df

,total_ct,landfalling
0.5,8756,3175
0.25,11616,3575
0.75,8122,3075
1.0,7740,3039


In [8]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.total_ct - res_df.loc['0.5'].total_ct)/res_df.loc['0.5'].total_ct

0.5     0.000000
0.25    0.326633
0.75   -0.072407
1.0    -0.116035
Name: total_ct, dtype: float64

In [10]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.landfalling - res_df.loc['0.5'].landfalling)/res_df.loc['0.5'].landfalling

0.5     0.000000
0.25    0.125984
0.75   -0.031496
1.0    -0.042835
Name: landfalling, dtype: float64

## epsilon_time perturbations

In [11]:
# constructing dictionaries to organize results
time_dict = {}
time_dict['9'] = pd.read_hdf(path + 'epsspace0.5_epstime9.0_minpts5_nreppts10_seed12345.h5')
time_dict['12'] = baseline_df
time_dict['15'] = pd.read_hdf(path + 'epsspace0.5_epstime15.0_minpts5_nreppts10_seed12345.h5')
time_dict['18'] = pd.read_hdf(path + 'epsspace0.5_epstime18.0_minpts5_nreppts10_seed12345.h5')
time_dict['21'] = pd.read_hdf(path + 'epsspace0.5_epstime21.0_minpts5_nreppts10_seed12345.h5')
time_dict['24'] = pd.read_hdf(path + 'epsspace0.5_epstime24.0_minpts5_nreppts10_seed12345.h5')

In [23]:
res_df = process_cts(time_dict)
res_df

,total_ct,landfalling
9,9316,3303
12,8756,3175
15,8374,3071
18,8052,2984
21,7897,2930
24,7637,2878


In [16]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.total_ct - res_df.loc['12'].total_ct)/res_df.loc['12'].total_ct

9     0.063956
12    0.000000
15   -0.043627
18   -0.080402
21   -0.098104
24   -0.127798
Name: total_ct, dtype: float64

In [18]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.landfalling - res_df.loc['12'].landfalling)/res_df.loc['12'].landfalling

9     0.040315
12    0.000000
15   -0.032756
18   -0.060157
21   -0.077165
24   -0.093543
Name: landfalling, dtype: float64

## minpts perturbations

In [19]:
minpts_dict = {}
minpts_dict['3'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts3_nreppts10_seed12345.h5')
minpts_dict['5'] = baseline_df
minpts_dict['8'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts8_nreppts10_seed12345.h5')
minpts_dict['10'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts10_nreppts10_seed12345.h5')

In [24]:
res_df = process_cts(minpts_dict)
res_df

,total_ct,landfalling
3,8907,3193
5,8756,3175
8,8181,3038
10,7858,2939


In [25]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.total_ct - res_df.loc['5'].total_ct)/res_df.loc['5'].total_ct

3     0.017245
5     0.000000
8    -0.065669
10   -0.102558
Name: total_ct, dtype: float64

In [26]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.landfalling - res_df.loc['5'].landfalling)/res_df.loc['5'].landfalling

3     0.005669
5     0.000000
8    -0.043150
10   -0.074331
Name: landfalling, dtype: float64

## n_rep_pts perturbations

In [27]:
reppts_dict = {}
reppts_dict['5'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts5_nreppts5_seed12345.h5')
reppts_dict['10'] = baseline_df
reppts_dict['15'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts5_nreppts15_seed12345.h5')

In [28]:
res_df = process_cts(reppts_dict)
res_df

,total_ct,landfalling
5,8685,3116
10,8756,3175
15,8643,3162


In [32]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.total_ct - res_df.loc['10'].total_ct)/res_df.loc['10'].total_ct

5    -0.008109
10    0.000000
15   -0.012905
Name: total_ct, dtype: float64

In [31]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.landfalling - res_df.loc['10'].landfalling)/res_df.loc['10'].landfalling

5    -0.018583
10    0.000000
15   -0.004094
Name: landfalling, dtype: float64

## seed perturbations

We also have a random component to this clustering algorithm. What if we switch up our random seed, how sensitive are the results?

In [33]:
seed_dict = {}
seed_dict['1111'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts5_nreppts10_seed1111.h5')
seed_dict['12345'] = baseline_df
seed_dict['2222'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts5_nreppts10_seed2222.h5')
seed_dict['3333'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts5_nreppts10_seed3333.h5')
seed_dict['4444'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts5_nreppts10_seed4444.h5')
seed_dict['5555'] = pd.read_hdf(path + 'epsspace0.5_epstime12_minpts5_nreppts10_seed5555.h5')

In [34]:
res_df = process_cts(seed_dict)
res_df

,total_ct,landfalling
1111,8754,3162
12345,8756,3175
2222,8760,3149
3333,8757,3164
4444,8734,3180
5555,8739,3167


In [35]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.total_ct - res_df.loc['12345'].total_ct)/res_df.loc['12345'].total_ct

1111    -0.000228
12345    0.000000
2222     0.000457
3333     0.000114
4444    -0.002513
5555    -0.001942
Name: total_ct, dtype: float64

In [36]:
# percent difference, expressed as a percentage of the total number of storms in baseline configuration
(res_df.landfalling - res_df.loc['12345'].landfalling)/res_df.loc['12345'].landfalling

1111    -0.004094
12345    0.000000
2222    -0.008189
3333    -0.003465
4444     0.001575
5555    -0.002520
Name: landfalling, dtype: float64